# CareerPilot AI — 3. Job Matching (RAG retrieval)

Loads the most recently saved profile from `careerpilot.db` and the postings from `job_dataset.json`, embeds both with a pre-trained model, and ranks jobs by cosine similarity. This is retrieval, not training — nothing here learns anything, we're just comparing pre-trained embeddings.

**Requires:** `1_resume_parsing.ipynb` has been run at least once (so `careerpilot.db` has a profile), and `2_job_fetching.ipynb` has been run at least once (so `job_dataset.json` exists).

**Output:** prints the top 5 matches and saves them to `top_matches.json`, which `4_generate_explanations.ipynb` reads next.


## Step 1 — Setup


In [2]:
!pip install sentence-transformers --quiet


In [3]:
import json
import sqlite3
import numpy as np
from numpy.linalg import norm
from sentence_transformers import SentenceTransformer

DB_PATH = "careerpilot.db"
JOB_DATASET_PATH = "job_dataset.json"
TOP_MATCHES_PATH = "top_matches.json"

## Step 2 — Load the latest saved profile from SQLite


In [4]:
def load_latest_profile() -> dict:
    conn = sqlite3.connect(DB_PATH)
    try:
        cur = conn.cursor()
        cur.execute("SELECT * FROM career_profiles ORDER BY id DESC LIMIT 1")
        row = cur.fetchone()
        cols = [d[0] for d in cur.description]
    finally:
        conn.close()

    if row is None:
        raise ValueError("No profiles found in careerpilot.db — run 1_resume_parsing.ipynb first.")

    profile = dict(zip(cols, row))
    for f in ["skills", "education", "experience", "organizations", "certifications"]:
        profile[f] = json.loads(profile[f]) if profile[f] else []
    return profile

profile = load_latest_profile()
profile_id = profile["id"]
print(f"Loaded profile #{profile_id}: {profile.get('name') or profile.get('filename')}")
print(f"Skills: {profile['skills']}")


Loaded profile #10: SOMISETTY SIVATEJA
Skills: ['CSS', 'FastAPI', 'Git', 'Java', 'JavaScript', 'Machine Learning', 'Python']


## Step 3 — Load job postings


In [5]:
with open(JOB_DATASET_PATH, "r") as f:
    jobs = json.load(f)

print(f"Loaded {len(jobs)} job postings.")


Loaded 34 job postings.


## Step 4 — Embed and rank by cosine similarity

`all-MiniLM-L6-v2` downloads once (a couple hundred MB, may take a minute or two the first time) — don't run other cells while it's working.


In [6]:
def profile_to_text(p: dict) -> str:
    skills_text = ", ".join(p.get("skills", []))
    return f"Skills: {skills_text}. Background: {p.get('raw_text', '')[:1000]}"

def job_to_text(job: dict) -> str:
    skills_text = ", ".join(job.get("skills", []))
    return f"{job['title']} at {job['company']}. Required skills: {skills_text}. {job['description']}"

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (norm(a) * norm(b)))

model = SentenceTransformer("all-MiniLM-L6-v2")

profile_vector = model.encode(profile_to_text(profile))
job_texts = [job_to_text(job) for job in jobs]
job_vectors = model.encode(job_texts)

scored = [(cosine_similarity(profile_vector, vec), job) for job, vec in zip(jobs, job_vectors)]
scored.sort(key=lambda x: x[0], reverse=True)
top_matches = scored[:5]

print("Top matches:\n")
for rank, (score, job) in enumerate(top_matches, start=1):
    print(f"{rank}. {job['title']} — {job['company']}  (score: {score:.3f})")
    print(f"   Required skills: {', '.join(job['skills'])}")
    print(f"   Apply: {job.get('apply_link', 'link not available')}")
    print()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Top matches:

1. Python Developer — Weekday AI  (score: 0.630)
   Required skills: Python
   Apply: https://www.adzuna.in/details/5636067155?utm_medium=api&utm_source=6abd39fa

2. Assistant Professor of Computer Science — SRM Institute of Science and Technology, Kattankulathur Campus  (score: 0.623)
   Required skills: Deep Learning, Machine Learning
   Apply: https://www.adzuna.in/land/ad/5742257783?se=DGRXs1CH8RGlu8FcJJTwKw&utm_medium=api&utm_source=6abd39fa&v=5B2079667667CB7E6004F6E1E1A71D70D49BBB13

3. Python Developer — Vrinda International  (score: 0.585)
   Required skills: Python
   Apply: https://www.adzuna.in/details/5786738445?utm_medium=api&utm_source=6abd39fa

4. Python Developer — Deqode  (score: 0.583)
   Required skills: FastAPI, Git, MongoDB, PostgreSQL, Python, SQL
   Apply: https://www.adzuna.in/details/5710913309?utm_medium=api&utm_source=6abd39fa

5. Faculty Member (as per UGC Cadre) - Computer Science, AI & Machine Learning — ATLAS SkillTech University  (score: 0.

## Step 5 — Save results for the explanation step


In [7]:
output = {
    "profile_id": profile_id,
    "matches": [
        {"score": score, **job} for score, job in top_matches
    ],
}

with open(TOP_MATCHES_PATH, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved {TOP_MATCHES_PATH} — ready for 4_generate_explanations.ipynb")

Saved top_matches.json — ready for 4_generate_explanations.ipynb
